# Car License Plates Detection and Recognition

Two-stage Chinese license plate detection and recognition on CCPD2019, combining YOLOv8n with a seven-head residual CNN.

**Team:** Haoqing Gu, Junwen Xue, Zezheng Yu, Ye Li, Xiaoyang Chen.

**Start here:** [README](README.md) for setup, [project overview](PROJECT_OVERVIEW.html) for saved results, and [project report](report/Project_Report.pdf) for the full analysis. This is a course group project; the PDF uses a conference-style format.

The notebook is supplied without cached outputs. Saved CSVs, figures and final checkpoints are included separately. Full reruns require the external CCPD2019 image dataset.


# Part 1: Data Preparation

The project uses the [CCPD2019 dataset](https://github.com/detectRecog/CCPD). It contains road images with one Chinese licence plate in each image. The plate location, four corner points, seven character labels and several image properties are stored in the filename.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent)
                    if (p / "config.py").is_file()
                    and (p / "detection").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open the notebook from the project root.")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
required_modules = ("torch", "torchvision", "ultralytics", "cv2", "pandas",
                    "numpy", "yaml", "PIL", "matplotlib", "IPython")
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError("Missing packages: " + ", ".join(missing)
                      + ". Install requirements.txt in this notebook's Python environment.")
CACHE_DIR = PROJECT_DIR / ".cache"
CACHE_DIR.mkdir(exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(CACHE_DIR)
os.environ["MPLCONFIGDIR"] = str(CACHE_DIR / "matplotlib")
import torch
import ultralytics
if torch.cuda.is_available():
    DEVICE, DEVICE_NAME = "0", torch.cuda.get_device_name(0)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE, DEVICE_NAME = "mps", "Apple Metal (MPS)"
else:
    DEVICE, DEVICE_NAME = "cpu", "CPU"
print("Project:", PROJECT_DIR)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("Device:", DEVICE_NAME)


In [ ]:
from config import DATA_ROOT, DATASET_VARIANT, SPLIT_DIR, OUT_DIR, SEED, N_TRAIN, N_VAL

print("DATA_ROOT:", DATA_ROOT)
print("SPLIT_DIR:", SPLIT_DIR)
print("OUT_DIR:", OUT_DIR)
print("DATASET_VARIANT:", DATASET_VARIANT)
assert DATASET_VARIANT == "ccpd2019", "This combined notebook requires CCPD2019."

## Reading CCPD filenames

Each filename stores the plate area, tilt, bounding box, four vertices, character indices, brightness, and blur. The parser returns the vertices in clockwise order: TL, TR, BR, BL.

In [ ]:
from parse_tool import parse_filename, PROVINCES, ALPHABETS, ADS

name = ("025-95_113-154&383_386&473-386&473_177&454_154&383_363&402"
        "-0_0_22_27_27_33_16-37-15.jpg")
r = parse_filename(name)
assert r["plate_code"] == PROVINCES[0] + "AY339S"
assert r["bbox"] == [154, 383, 386, 473]
assert r["four_points"][0] == (154, 383)
assert r["bright"] == 37 and r["blur"] == 15
print("Self-test passed: character indices, geometry, brightness, and blur decoded correctly.")

## Dataset splits

Seed 9444 fixes every split. The training list contains 30,000 images. A shuffled copy of `val.txt` supplies 5,000 validation images and a separate 3,000-image Base test set.

The 3,000-image development list is drawn from training. Six difficult test sets come from the official lists, while Weather is sampled from its image directory because CCPD does not provide a Weather list.

In [ ]:
import os
import random
from config import DATA_ROOT, SPLIT_DIR, OUT_DIR, SEED, N_TRAIN, N_VAL
from parse_tool import parse_filename

def read_list(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return [ln.strip() for ln in f if ln.strip()]

def write_list(rel_paths, name):
    out = os.path.join(OUT_DIR, name)
    with open(out, "w", encoding="utf-8") as f:
        f.write("\n".join(rel_paths) + "\n")
    print(f"{name}: {len(rel_paths)}")

train_all = read_list(os.path.join(SPLIT_DIR, "train.txt"))
base_val_all = read_list(os.path.join(SPLIT_DIR, "val.txt"))
assert len(train_all) >= N_TRAIN
assert len(base_val_all) >= N_VAL + 3000

train_sub = random.Random(SEED).sample(train_all, N_TRAIN)
base_pool = base_val_all.copy()
random.Random(SEED + 1).shuffle(base_pool)
val_sub = base_pool[:N_VAL]
base_test_sub = base_pool[N_VAL:N_VAL + 3000]
dev_sub = random.Random(SEED + 2).sample(train_sub, 3000)

write_list(train_sub, "train_30k.txt")
write_list(val_sub, "val_5k.txt")
write_list(dev_sub, "train_dev_3k.txt")
write_list(base_test_sub, "test_base_3k.txt")

difficult_subsets = ["blur", "challenge", "db", "fn", "rotate", "tilt"]
for offset, subset in enumerate(difficult_subsets, start=2):
    candidates = read_list(os.path.join(SPLIT_DIR, f"ccpd_{subset}.txt"))
    assert len(candidates) >= 1000
    selected = random.Random(SEED + offset).sample(candidates, 1000)
    write_list(selected, f"test_{subset}_1k.txt")

weather_dir = os.path.join(DATA_ROOT, "ccpd_weather")
weather_candidates = [
    f"ccpd_weather/{filename}"
    for filename in sorted(os.listdir(weather_dir))
    if filename.lower().endswith(".jpg")
]
assert len(weather_candidates) >= 1000
weather_sub = random.Random(SEED + 8).sample(weather_candidates, 1000)
write_list(weather_sub, "test_weather_1k.txt")

## Split checks

Training, validation, and test paths are checked for duplicates and overlap.

`train_dev_3k` is part of the training set and is not treated as an independent evaluation split.

In [ ]:
def normalise_path(path):
    return os.path.normcase(os.path.normpath(path))


def path_set(paths):
    return {normalise_path(path) for path in paths}


formal_files = {
    "train": "train_30k.txt",
    "validation": "val_5k.txt",
    "base": "test_base_3k.txt",
    "blur": "test_blur_1k.txt",
    "challenge": "test_challenge_1k.txt",
    "db": "test_db_1k.txt",
    "fn": "test_fn_1k.txt",
    "rotate": "test_rotate_1k.txt",
    "tilt": "test_tilt_1k.txt",
    "weather": "test_weather_1k.txt",
}
formal_paths = {
    name: read_list(os.path.join(OUT_DIR, filename))
    for name, filename in formal_files.items()
}
formal_sets = {name: path_set(paths) for name, paths in formal_paths.items()}

for name, paths in formal_paths.items():
    assert len(paths) == len(formal_sets[name]), f"Duplicate paths inside {name}"

names = list(formal_sets)
for first_index, first in enumerate(names):
    for second in names[first_index + 1:]:
        overlap = formal_sets[first] & formal_sets[second]
        print(f"{first} vs {second}: {len(overlap)}")
        assert not overlap, f"{first} and {second} overlap"

dev_set = path_set(read_list(os.path.join(OUT_DIR, "train_dev_3k.txt")))
assert dev_set.issubset(formal_sets["train"]), "Development set must come from training"
print("All formal datasets are pairwise disjoint; the development set is inside training.")

## Image and annotation checks

Every selected image is decoded and checked against its filename annotation. Boundary coordinates equal to the image width or height are accepted because they represent the exclusive edge of the image.

In [ ]:
import cv2

def validate_dataset_list(display_name, filename):
    paths = read_list(
        os.path.join(OUT_DIR, filename)
    )

    missing_files = []
    unreadable_files = []
    invalid_annotations = []
    out_of_bounds_annotations = []

    for rel_path in paths:
        full_path = (
            rel_path
            if os.path.isabs(rel_path)
            else os.path.join(DATA_ROOT, rel_path)
        )

        if not os.path.isfile(full_path):
            missing_files.append(rel_path)
            continue

        image = cv2.imread(full_path)
        if image is None:
            unreadable_files.append(rel_path)
            continue

        try:
            info = parse_filename(rel_path)
        except Exception as error:
            invalid_annotations.append(
                (rel_path, str(error))
            )
            continue

        height, width = image.shape[:2]
        x1, y1, x2, y2 = info["bbox"]
        bbox_valid = 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height
        points_valid = all(
            0 <= x <= width and 0 <= y <= height
            for x, y in info["four_points"]
        )
        if not (bbox_valid and points_valid):
            out_of_bounds_annotations.append(rel_path)

    valid_count = (
        len(paths)
        - len(missing_files)
        - len(unreadable_files)
        - len(invalid_annotations)
        - len(out_of_bounds_annotations)
    )

    print(f"\n{display_name}")
    print("total:", len(paths))
    print("valid:", valid_count)
    print("missing:", len(missing_files))
    print("unreadable:", len(unreadable_files))
    print("invalid:", len(invalid_annotations))
    print("out of bounds:", len(out_of_bounds_annotations))

    return {
        "name": display_name,
        "total": len(paths),
        "valid": valid_count,
        "missing": missing_files,
        "unreadable": unreadable_files,
        "invalid": invalid_annotations,
        "out_of_bounds": out_of_bounds_annotations,
    }


selected_lists = {
    "Train": "train_30k.txt",
    "Validation": "val_5k.txt",
    "Base test": "test_base_3k.txt",
    "Blur test": "test_blur_1k.txt",
    "Challenge test": "test_challenge_1k.txt",
    "DB test": "test_db_1k.txt",
    "FN test": "test_fn_1k.txt",
    "Rotate test": "test_rotate_1k.txt",
    "Tilt test": "test_tilt_1k.txt",
    "Weather test": "test_weather_1k.txt",
}

validation_results = []

for display_name, filename in selected_lists.items():
    result = validate_dataset_list(
        display_name,
        filename
    )
    validation_results.append(result)

failed_lists = [result for result in validation_results if result["valid"] != result["total"]]
assert not failed_lists, f"Integrity checks failed for {[item['name'] for item in failed_lists]}"
print("All selected images and annotations passed the integrity checks.")

## Metadata

One CSV stores the image paths, boxes, vertices, character indices, plate text, image attributes, and split labels used by both models.

In [ ]:
import os
import json
import pandas as pd

def build_metadata_record(rel_path, split, subset):
    info = parse_filename(rel_path)

    x1, y1, x2, y2 = info["bbox"]

    filename = os.path.splitext(
        os.path.basename(rel_path)
    )[0]

    parts = filename.split("-")
    plate_indices = list(
        map(int, parts[4].split("_"))
    )

    return {
        "split": split,
        "subset": subset,
        "relative_path": rel_path,
        "scale": info["scale"],
        "horizontal_tilt": info["h_tilt"],
        "vertical_tilt": info["v_tilt"],
        "x_min": x1,
        "y_min": y1,
        "x_max": x2,
        "y_max": y2,
        "four_points": json.dumps(
            info["four_points"],
            ensure_ascii=False
        ),
        "plate_indices": "_".join(
            map(str, plate_indices)
        ),
        "plate_text": info["plate_code"],
        "province": info["province"],
        "number_of_characters": info["n_chars"],
        "brightness": info["bright"],
        "blur": info["blur"],
    }


metadata_sources = [
    ("train", "base", "train_30k.txt"),
    ("validation", "base", "val_5k.txt"),
    ("test", "base", "test_base_3k.txt"),
    ("test", "blur", "test_blur_1k.txt"),
    ("test", "challenge", "test_challenge_1k.txt"),
    ("test", "db", "test_db_1k.txt"),
    ("test", "fn", "test_fn_1k.txt"),
    ("test", "rotate", "test_rotate_1k.txt"),
    ("test", "tilt", "test_tilt_1k.txt"),
    ("test", "weather", "test_weather_1k.txt"),
]

records = []

for split, subset, filename in metadata_sources:
    paths = read_list(
        os.path.join(OUT_DIR, filename)
    )

    for rel_path in paths:
        records.append(
            build_metadata_record(
                rel_path,
                split,
                subset
            )
        )

metadata_df = pd.DataFrame(records)
assert len(metadata_df) == 45000, f"Expected 45,000 metadata rows, got {len(metadata_df)}"
assert not metadata_df["relative_path"].duplicated().any(), "Metadata contains duplicate images"
assert metadata_df["number_of_characters"].eq(7).all(), "Selected data contains non-seven-character plates"

metadata_path = os.path.join(
    OUT_DIR,
    "ccpd_selected_metadata.csv"
)

metadata_df.to_csv(
    metadata_path,
    index=False,
    encoding="utf-8-sig"
)

print("Metadata saved to:", metadata_path)
print("Metadata rows:", len(metadata_df))

metadata_df.head()

In [ ]:
dataset_summary = (
    metadata_df
    .groupby(["split", "subset"])
    .size()
    .reset_index(name="number_of_images")
)

def position_class_sets(frame):
    classes = [set() for _ in range(7)]
    for encoded_indices in frame["plate_indices"]:
        values = list(map(int, encoded_indices.split("_")))
        for position, value in enumerate(values):
            classes[position].add(value)
    return classes

train_classes = position_class_sets(metadata_df[metadata_df["split"] == "train"])
evaluation_classes = position_class_sets(metadata_df[metadata_df["split"] != "train"])
coverage_records = []
for position, (seen, evaluated) in enumerate(zip(train_classes, evaluation_classes), start=1):
    coverage_records.append({
        "position": position,
        "training_classes": len(seen),
        "evaluation_classes": len(evaluated),
        "unseen_evaluation_classes": sorted(evaluated - seen),
    })

coverage_df = pd.DataFrame(coverage_records)
assert not coverage_df["unseen_evaluation_classes"].map(bool).any(), (
    "At least one evaluation class is absent from training"
)
display(dataset_summary)
coverage_df

The final dataset contains 30,000 training, 5,000 validation, and 10,000 test images. Every character class found in validation or testing also appears in training.

## Exploratory analysis

Filename attributes and character counts use all 30,000 training images. Image-based plots use a fixed sample of 3,000.

In [ ]:
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

from config import OUT_DIR, SEED
from parse_tool import parse_filename

random.seed(SEED)
np.random.seed(SEED)

train_list = read_list(
    os.path.join(OUT_DIR, "train_30k.txt")
)

stats_paths = train_list
infos = [parse_filename(path) for path in stats_paths]

visual_paths = random.sample(
    train_list,
    min(3000, len(train_list))
)

print("Training images:", len(train_list))
print("Images used for attribute and character statistics:", len(stats_paths))
print("Images used for decoded-image visualisation:", len(visual_paths))

specs = [
    ("scale", "Plate Area Ratio", "Area ratio"),
    ("h_tilt", "Horizontal Tilt", "Tilt degree"),
    ("v_tilt", "Vertical Tilt", "Tilt degree"),
    ("bright", "Brightness", "Brightness value"),
    ("blur", "Blurriness", "Blur value"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (key, title, x_label) in zip(axes, specs):
    values = [info[key] for info in infos]

    ax.hist(
        values,
        bins=25,
        alpha=0.85
    )

    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Number of images")
    ax.grid(axis="y", alpha=0.3)

for index in range(len(specs), len(axes)):
    axes[index].axis("off")

plt.suptitle(
    f"Attribute Distributions ({len(infos):,} Training Images)",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Most plates occupy only a small part of the image, making them relatively small detection targets.

The training set also contains variation in tilt, brightness, and blur.

In [ ]:
from collections import Counter

province_indices = [int(Path(path).stem.split("-")[4].split("_")[0]) for path in stats_paths]
prov_cnt = Counter(province_indices)
provs, pcs = zip(*sorted(prov_cnt.items()))

plt.figure(figsize=(10, 4))
plt.bar(range(len(provs)), pcs, color="#EA4335")
plt.xticks(range(len(provs)), [str(value) for value in provs])
plt.yscale("log")
plt.title("Province Class-Index Distribution (log scale)")
plt.xlabel("Province class index")
plt.ylabel("Count (log)")
plt.show()
print(f"Province classes seen: {len(prov_cnt)}; largest-class share: {100*max(pcs)/len(infos):.1f}%")

char_cnt = Counter(c for i in infos for c in i["plate_code"][1:])
chars, ccs = zip(*sorted(char_cnt.items()))
plt.figure(figsize=(12, 3.5))
plt.bar(chars, ccs, color="#FBBC05")
plt.title("Letter/Digit Frequency (positions 2-7)")
plt.show()

The province distribution is heavily imbalanced. Position-wise accuracy is needed alongside overall recognition accuracy.

In [ ]:
test_names = ["base", "db", "blur", "fn", "rotate", "tilt", "challenge", "weather"]
fig, axes = plt.subplots(2, 8, figsize=(22, 7))
for col, sub in enumerate(test_names):
    rels = read_list(os.path.join(OUT_DIR, f"test_{sub}_1k.txt" if sub != "base" else "test_base_3k.txt"))
    for row, rel in enumerate(random.sample(rels, 2)):
        img = cv2.imread(os.path.join(DATA_ROOT, rel))
        info = parse_filename(rel)
        x1, y1, x2, y2 = info["bbox"]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 6)
        axes[row][col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if row == 0:
            axes[row][col].set_title(sub, fontsize=11)
        axes[row][col].axis("off")
plt.suptitle("Random samples per test subset (GT box in red)")
plt.tight_layout()
plt.show()

These samples show the conditions represented by each test subset. Red rectangles mark the ground-truth boxes.

In [ ]:
demo_rel = random.choice(read_list(os.path.join(OUT_DIR, "test_rotate_1k.txt")))
img = cv2.imread(os.path.join(DATA_ROOT, demo_rel))
info = parse_filename(demo_rel)
x1, y1, x2, y2 = info["bbox"]
cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 5)
crop = img[y1:y2, x1:x2]
PLATE_WIDTH = 192
PLATE_HEIGHT = 48
resized_crop = cv2.resize(crop, (PLATE_WIDTH, PLATE_HEIGHT))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, im, title in zip(
    axes,
    [img, crop, resized_crop],
    ["Ground-truth box", "Bounding-box crop", "Resized crop (192 x 48)"],
):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

Recognition uses the complete bounding-box crop resized to 192 x 48, without character segmentation.

In [ ]:
heat = np.zeros((72, 116))
ws, hs = [], []
for i in infos:
    x1, y1, x2, y2 = i["bbox"]
    cx, cy = (x1 + x2) // 20, (y1 + y2) // 20
    if 0 <= cy < heat.shape[0] and 0 <= cx < heat.shape[1]:
        heat[cy, cx] += 1
    ws.append(x2 - x1)
    hs.append(y2 - y1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(heat, cmap="hot")
axes[0].set_title("Plate Center Heatmap (720x1160 image)")
axes[1].scatter(ws, hs, s=3, alpha=0.3, color="#4285F4")
axes[1].set_xlabel("plate width (px)")
axes[1].set_ylabel("plate height (px)")
axes[1].set_title("Plate Size Distribution")
plt.tight_layout()
plt.show()

Plate position and size vary across the training set.

In [ ]:
check_dir = os.path.join(OUT_DIR, "anno_check")
os.makedirs(check_dir, exist_ok=True)
saved_checks = []
for idx, rel in enumerate(random.sample(train_list, 20)):
    img = cv2.imread(os.path.join(DATA_ROOT, rel))
    info = parse_filename(rel)
    x1, y1, x2, y2 = info["bbox"]
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 3)
    for j, (px, py) in enumerate(info["four_points"]):
        cv2.circle(img, (px, py), 8, (0, 255, 0), -1)
        cv2.putText(img, str(j), (px + 8, py - 8), 0, 0.9, (0, 255, 0), 2)
    output_path = os.path.join(check_dir, f"check_{idx:02d}.png")
    cv2.imwrite(output_path, img)
    saved_checks.append(output_path)

print("Saved 20 annotation-check images to", check_dir)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for axis, path in zip(axes.flat, saved_checks[:6]):
    image = cv2.imread(path)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(Path(path).name)
    axis.axis("off")
plt.suptitle("Saved annotation checks (also displayed inline)")
plt.tight_layout()
plt.show()

# Part 2: License Plate Detection

## Detection setup

YOLOv8n is trained as a one-class plate detector using 640 x 640 images. It was selected because it is a small model and gives a useful balance between accuracy and speed. Training uses the 30,000-image split from Part 1, with 5,000 images for validation. The initial weights come from the [Ultralytics YOLOv8 implementation](https://github.com/ultralytics/ultralytics). The label conversion, fine-tuning and CCPD evaluation are carried out in this project.

Only the highest-confidence box is evaluated for each image. A detection is correct when its IoU with the ground-truth box is above 0.7. Precision, recall, mAP and inference speed are also reported.

YOLO label conversion for image width $W$ and height $H$:

$$c_x=\frac{x_1+x_2}{2W},\quad c_y=\frac{y_1+y_2}{2H},\quad w=\frac{x_2-x_1}{W},\quad h=\frac{y_2-y_1}{H}$$

Each label line is `0 c_x c_y w h`.

In [ ]:
PART2_DIR = PROJECT_DIR / "detection"
if str(PART2_DIR) not in sys.path:
    sys.path.insert(0, str(PART2_DIR))

from common import RESULTS_DIR, RUNS_DIR, YOLO_DATASET_DIR

print("PART2_DIR:", PART2_DIR)
print("YOLO_DATASET_DIR:", YOLO_DATASET_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## YOLO data

The bounding boxes from Part 1 are converted to YOLO labels. Hard links avoid copying the image data when the file system supports them.

In [ ]:
full_yaml = YOLO_DATASET_DIR / "ccpd_full.yaml"
dev_yaml = YOLO_DATASET_DIR / "ccpd_dev.yaml"
required_dirs = [
    YOLO_DATASET_DIR / "images" / "train",
    YOLO_DATASET_DIR / "labels" / "train",
    YOLO_DATASET_DIR / "images" / "val",
    YOLO_DATASET_DIR / "labels" / "val",
]

if full_yaml.is_file() and dev_yaml.is_file() and all(path.is_dir() for path in required_dirs):
    print("Prepared YOLO dataset found; reusing it.")
else:
    subprocess.check_call(
        [sys.executable, str(PART2_DIR / "prepare_yolo_dataset.py"), "--mode", "auto"]
    )

print("Full YAML:", full_yaml)
print("Dev YAML:", dev_yaml)

## Label check

In [ ]:
from PIL import Image
import matplotlib.patches as patches

sample_images = sorted((YOLO_DATASET_DIR / "images" / "train").rglob("*.jpg"))[:3]
fig, axes = plt.subplots(1, 3, figsize=(14, 6))
for axis, image_path in zip(axes, sample_images):
    label_path = Path(str(image_path).replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}")).with_suffix(".txt")
    with Image.open(image_path) as image:
        width, height = image.size
        axis.imshow(image.convert("RGB"))
    class_id, cx, cy, bw, bh = map(float, label_path.read_text().split())
    x1, y1 = (cx - bw / 2) * width, (cy - bh / 2) * height
    axis.add_patch(patches.Rectangle((x1, y1), bw * width, bh * height, fill=False, edgecolor="red", linewidth=3))
    axis.set_title(image_path.name[:20] + "...")
    axis.axis("off")
plt.suptitle("YOLO labels after conversion")
plt.tight_layout()
plt.show()

## Detector training

The detector was trained for 30 epochs on all 30,000 training images. The saved best checkpoint is used for the remaining detection experiments.

In [ ]:
MODEL_SOURCE = str(PART2_DIR / "yolov8n.pt") if (PART2_DIR / "yolov8n.pt").is_file() else "yolov8n.pt"
FULL_EPOCHS = 30
FULL_RUN_NAME = "ccpd2019_yolov8n_gpu_30ep"
FULL_RUN_DIR = RUNS_DIR / FULL_RUN_NAME
FULL_WEIGHTS = FULL_RUN_DIR / "weights" / "best.pt"

def run_is_complete(run_dir, expected_epochs):
    csv_path = run_dir / "results.csv"
    weights_path = run_dir / "weights" / "best.pt"
    if not csv_path.is_file() or not weights_path.is_file():
        return False
    return len(pd.read_csv(csv_path)) >= expected_epochs

if not run_is_complete(FULL_RUN_DIR, FULL_EPOCHS):
    if not torch.cuda.is_available():
        raise RuntimeError("Detector retraining requires CUDA for the recorded setup. The final checkpoint is included.")
    torch.cuda.empty_cache()
    subprocess.check_call([
        sys.executable, str(PART2_DIR / "train_detector.py"),
        "--data", str(full_yaml), "--model", MODEL_SOURCE,
        "--epochs", str(FULL_EPOCHS), "--imgsz", "640", "--batch", "16",
        "--device", "0", "--name", FULL_RUN_NAME,
    ])
else:
    print("Completed 30-epoch detector found:", FULL_WEIGHTS)

WEIGHTS = FULL_WEIGHTS
assert WEIGHTS.is_file(), f"Training did not create {WEIGHTS}"
print("Evaluation weights:", WEIGHTS)

## Training curves

In [ ]:
from IPython.display import Image as NotebookImage, display

training_dir = WEIGHTS.parent.parent
training_csv = training_dir / "results.csv"
if not training_csv.is_file():
    raise FileNotFoundError(f"Training history was not found: {training_csv}")

training_history = pd.read_csv(training_csv)
training_history.columns = training_history.columns.str.strip()
display(training_history.tail(10))

map_metric = "metrics/mAP50-95(B)"
map50_metric = "metrics/mAP50(B)"
best_index = training_history[map_metric].idxmax()
best_row = training_history.loc[best_index]

print("Best epoch:", int(best_row["epoch"]))
print("Best validation mAP50-95:", f"{best_row[map_metric]:.4f}")
print("Final validation mAP50-95:", f"{training_history.iloc[-1][map_metric]:.4f}")
print("Final validation mAP50:", f"{training_history.iloc[-1][map50_metric]:.4f}")

results_image = training_dir / "results.png"
if results_image.is_file():
    display(NotebookImage(filename=str(results_image)))

epochs = training_history["epoch"].astype(int)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, training_history["train/box_loss"], label="Training box loss")
axes[0].plot(epochs, training_history["val/box_loss"], label="Validation box loss")
axes[0].set(title="License-plate localisation loss", xlabel="Epoch", ylabel="Box loss")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].plot(epochs, training_history[map_metric], color="tab:green", label="Validation mAP50-95")
axes[1].scatter(int(best_row["epoch"]), best_row[map_metric], color="red", label="Best checkpoint")
axes[1].set(title="Strict validation detection performance", xlabel="Epoch", ylabel="mAP50-95")
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

print(
    "mAP50 is expected to be high for this one-class task; mAP50-95 and "
    "Accuracy@IoU>0.7 provide stricter localisation evidence."
)

## Detection results

The final model is evaluated on the fixed Base, Blur, Challenge, DB, FN, Rotate, Tilt, and Weather test sets. Results include top-1 accuracy at IoU > 0.7, precision, recall and mAP.

In [ ]:
EVALUATION_NAME = "ccpd2019_gpu_30ep_original_seed"
summary_path = RESULTS_DIR / f"{EVALUATION_NAME}_summary.csv"

if summary_path.is_file():
    print("Existing detection results found:", summary_path)
else:
    subprocess.check_call([
        sys.executable, str(PART2_DIR / "evaluate_detector.py"),
        "--weights", str(WEIGHTS), "--subsets", "all",
        "--imgsz", "640", "--batch", "16", "--device", "0",
        "--with-map", "--output-name", EVALUATION_NAME,
    ])

metrics_df = pd.read_csv(summary_path)
metric_columns = [
    "subset", "images", "detections", "ccpd_accuracy_iou_gt_0_7",
    "mean_top1_iou", "precision", "recall", "mAP50", "mAP75", "mAP50_95",
]
display(metrics_df[[column for column in metric_columns if column in metrics_df.columns]].style.format({
    "ccpd_accuracy_iou_gt_0_7": "{:.2%}",
    "mean_top1_iou": "{:.4f}", "precision": "{:.4f}", "recall": "{:.4f}",
    "mAP50": "{:.4f}", "mAP75": "{:.4f}", "mAP50_95": "{:.4f}",
}))

overall_accuracy = metrics_df["ccpd_correct_iou_gt_0_7"].sum() / metrics_df["images"].sum()
print(f"Overall CCPD top-1 Accuracy @ IoU>0.7: {overall_accuracy:.2%}")

## Inference speed

In [ ]:
SPEED_NAME = f"{EVALUATION_NAME}_speed"
speed_path = RESULTS_DIR / f"{SPEED_NAME}.csv"
if speed_path.is_file():
    print("Existing speed result found:", speed_path)
else:
    subprocess.check_call([
        sys.executable, str(PART2_DIR / "benchmark_speed.py"),
        "--weights", str(WEIGHTS), "--subset", "base",
        "--num-images", "500", "--warmup", "20", "--imgsz", "640",
        "--device", "0", "--output-name", SPEED_NAME,
    ])

speed_df = pd.read_csv(speed_path)
display(speed_df)

## Detection examples

Each example shows the ground-truth box in red and the highest-confidence prediction in green.

In [ ]:
from ultralytics import YOLO

from common import full_image_path, list_file_for_subset, read_list

gallery_dir = RESULTS_DIR / "prediction_gallery"
gallery_dir.mkdir(parents=True, exist_ok=True)
subsets = ["base", "blur", "challenge", "db", "fn", "rotate", "tilt", "weather"]
gallery_paths = {subset: gallery_dir / f"{subset}.jpg" for subset in subsets}
detector = None

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for axis, subset in zip(axes.flat, subsets):
    output_path = gallery_paths[subset]
    if output_path.is_file():
        image = cv2.imread(str(output_path))
        caption = "GT red, prediction green"
    else:
        if detector is None:
            detector = YOLO(str(WEIGHTS))
        relative_path = sorted(read_list(list_file_for_subset(subset)))[0]
        image_path = full_image_path(relative_path)
        image = cv2.imread(str(image_path))
        gt = parse_filename(relative_path)["bbox"]
        cv2.rectangle(image, (gt[0], gt[1]), (gt[2], gt[3]), (0, 0, 255), 4)

        result = detector.predict(
            source=str(image_path), imgsz=640, conf=0.25, device=DEVICE,
            verbose=False, save=False,
        )[0]
        caption = "no prediction"
        if result.boxes is not None and len(result.boxes):
            confidences = result.boxes.conf.detach().cpu().numpy()
            best_index = int(np.argmax(confidences))
            x1, y1, x2, y2 = result.boxes.xyxy[best_index].detach().cpu().numpy().astype(int)
            confidence = float(confidences[best_index])
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 4)
            caption = f"confidence={confidence:.3f}"
        cv2.imwrite(str(output_path), image)

    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(f"{subset}: {caption}")
    axis.axis("off")

plt.suptitle("CCPD2019 detection gallery: GT red, prediction green")
plt.tight_layout()
plt.show()
print("Prediction images saved to:", gallery_dir)

# Part 3: License Plate Recognition


## Recognition setup

In [ ]:
import os
import random
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

cwd = Path.cwd().resolve()
PROJECT_DIR = next(
    (path for path in (cwd, cwd.parent) if (path / "config.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the project folder.")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from config import DATA_ROOT, OUT_DIR, SEED
from parse_tool import ADS, ALPHABETS, PROVINCES

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
PART3_DIR = PROJECT_DIR / "recognition"
RESULTS3_DIR = PART3_DIR / "results"
CHECKPOINT3_DIR = PART3_DIR / "checkpoints"
RESULTS3_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT3_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = Path(OUT_DIR) / "ccpd_selected_metadata.csv"
metadata_df = pd.read_csv(METADATA_PATH)
assert len(metadata_df) == 45000
assert metadata_df["number_of_characters"].eq(7).all()

print("Project:", PROJECT_DIR)
print("Metadata:", METADATA_PATH)
print("Device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print(metadata_df.groupby(["split", "subset"]).size())

## Plate crops

The recogniser uses the complete ground-truth plate box rather than separate character crops. Each crop is resized to 192 x 48. Colour and small geometric changes are used during training, but horizontal flipping is excluded because it reverses the text.

In [ ]:
PLATE_WIDTH = 192
PLATE_HEIGHT = 48
POSITION_TABLES = [PROVINCES, ALPHABETS, ADS, ADS, ADS, ADS, ADS]
NUM_CLASSES = [len(table) for table in POSITION_TABLES]
NORMALISE_MEAN = (0.485, 0.456, 0.406)
NORMALISE_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((PLATE_HEIGHT, PLATE_WIDTH)),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.03),
    transforms.RandomAffine(degrees=3, translate=(0.03, 0.06), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(NORMALISE_MEAN, NORMALISE_STD),
])

evaluation_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((PLATE_HEIGHT, PLATE_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(NORMALISE_MEAN, NORMALISE_STD),
])

def crop_bounding_box(image, bbox):
    height, width = image.shape[:2]
    x1, y1, x2, y2 = map(int, bbox)
    x1 = max(0, min(x1, width - 1))
    y1 = max(0, min(y1, height - 1))
    x2 = max(x1, min(x2, width - 1))
    y2 = max(y1, min(y2, height - 1))
    crop = image[y1:y2 + 1, x1:x2 + 1]
    if crop.size == 0:
        raise ValueError(f"Empty bounding-box crop: {bbox}")
    return crop

def parse_plate_indices(encoded):
    values = [int(value) for value in str(encoded).split("_")]
    if len(values) != 7:
        raise ValueError(f"Expected seven labels, got {values}")
    return values

class CCPDRecognitionDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image_path = Path(DATA_ROOT) / row["relative_path"]
        image = cv2.imread(str(image_path))
        if image is None:
            raise FileNotFoundError(f"Could not decode {image_path}")
        crop = crop_bounding_box(
            image,
            (row["x_min"], row["y_min"], row["x_max"], row["y_max"]),
        )
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        labels = torch.tensor(parse_plate_indices(row["plate_indices"]), dtype=torch.long)
        return self.transform(crop), labels, row["relative_path"]

train_df = metadata_df[metadata_df["split"].eq("train")].reset_index(drop=True)
validation_df = metadata_df[metadata_df["split"].eq("validation")].reset_index(drop=True)
test_frames = {
    subset: metadata_df[
        metadata_df["split"].eq("test") & metadata_df["subset"].eq(subset)
    ].reset_index(drop=True)
    for subset in ("base", "blur", "challenge", "db", "fn", "rotate", "tilt", "weather")
}

for frame_name, frame in {"train": train_df, "validation": validation_df, **test_frames}.items():
    for encoded in frame["plate_indices"]:
        labels = parse_plate_indices(encoded)
        assert all(0 <= label < NUM_CLASSES[position] for position, label in enumerate(labels)), frame_name

BATCH_SIZE = 64
NUM_WORKERS = 0
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    CCPDRecognitionDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    generator=loader_generator,
)
validation_loader = DataLoader(
    CCPDRecognitionDataset(validation_df, evaluation_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

images, labels, paths = next(iter(train_loader))
print("Image batch:", tuple(images.shape))
print("Label batch:", tuple(labels.shape))

In [ ]:
def labels_to_text(indices):
    return "".join(POSITION_TABLES[position][int(value)] for position, value in enumerate(indices))

def labels_to_display_text(indices):
    values = [int(value) for value in indices]
    suffix = "".join(
        POSITION_TABLES[position][values[position]]
        for position in range(1, 7)
    )
    return f"P{values[0]:02d}-{suffix}"

def tensor_to_rgb(image_tensor):
    mean = torch.tensor(NORMALISE_MEAN).view(3, 1, 1)
    std = torch.tensor(NORMALISE_STD).view(3, 1, 1)
    return (image_tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for axis, image, target in zip(axes.flat, images[:8], labels[:8]):
    axis.imshow(tensor_to_rgb(image))
    axis.set_title(labels_to_display_text(target))
    axis.axis("off")
plt.suptitle("Augmented bounding-box crops and ground-truth labels")
plt.tight_layout()
plt.show()

## Recognition model

Every plate in the selected data has seven labelled positions, so the recogniser uses seven output heads instead of splitting the image into separate characters. A residual CNN processes the complete crop and shares its visual features across all positions. The first head predicts the province, the second predicts the first letter, and the other five predict letters or digits. This model was implemented for this project and keeps the recognition stage simple enough to train on the available hardware.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.skip = (
            nn.Identity()
            if stride == 1 and in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        )
        self.activation = nn.ReLU(inplace=True)

    def forward(self, inputs):
        return self.activation(self.main(inputs) + self.skip(inputs))

class SevenHeadRecognizer(nn.Module):
    def __init__(self, class_counts):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 64),
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128),
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256),
            nn.AdaptiveAvgPool2d((1, 12)),
        )
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 12, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
        )
        self.heads = nn.ModuleList(nn.Linear(512, count) for count in class_counts)

    def forward(self, inputs):
        features = self.shared(self.features(inputs))
        return [head(features) for head in self.heads]

recognizer = SevenHeadRecognizer(NUM_CLASSES).to(DEVICE)
with torch.no_grad():
    sample_outputs = recognizer(images[:2].to(DEVICE))
assert [tuple(output.shape) for output in sample_outputs] == [
    (2, count) for count in NUM_CLASSES
]
print("Trainable parameters:", f"{sum(parameter.numel() for parameter in recognizer.parameters() if parameter.requires_grad):,}")
print("Output shapes:", [tuple(output.shape) for output in sample_outputs])

## Training

The best checkpoint is selected by validation exact-plate accuracy. Character and position accuracies are recorded alongside it. An existing checkpoint and training history are loaded instead of repeating the same training run.

In [ ]:
def make_province_weights(frame):
    encoded = np.stack(frame["plate_indices"].map(parse_plate_indices).to_numpy())
    frequencies = np.bincount(encoded[:, 0], minlength=NUM_CLASSES[0])
    present = frequencies > 0
    weights = np.zeros(NUM_CLASSES[0], dtype=np.float32)
    weights[present] = np.sqrt(frequencies[present].mean() / frequencies[present])
    weights[present] = np.clip(weights[present], 0.5, 3.0)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

province_weights = make_province_weights(train_df)
criteria = [
    nn.CrossEntropyLoss(weight=province_weights, label_smoothing=0.02),
    *[nn.CrossEntropyLoss(label_smoothing=0.02) for _ in range(6)],
]

def batch_loss(outputs, targets):
    return sum(
        criterion(outputs[position], targets[:, position])
        for position, criterion in enumerate(criteria)
    ) / 7

def run_epoch(model, loader, optimizer=None, collect_predictions=False):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_plates = 0
    total_characters = 0
    correct_characters = 0
    exact_plates = 0
    position_correct = torch.zeros(7, dtype=torch.long)
    prediction_rows = []
    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if training else None

    for inputs, targets, paths in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp):
                outputs = model(inputs)
                loss = batch_loss(outputs, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scaler.step(optimizer)
                scaler.update()

        predictions = torch.stack([output.argmax(dim=1) for output in outputs], dim=1)
        matches = predictions.eq(targets)
        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_plates += batch_size
        total_characters += targets.numel()
        correct_characters += int(matches.sum())
        exact_plates += int(matches.all(dim=1).sum())
        position_correct += matches.sum(dim=0).cpu()

        if collect_predictions:
            for path, target, prediction, match in zip(
                paths, targets.cpu(), predictions.cpu(), matches.cpu()
            ):
                prediction_rows.append({
                    "relative_path": path,
                    "ground_truth": labels_to_text(target),
                    "prediction": labels_to_text(prediction),
                    "errors": int((~match).sum()),
                    "ground_truth_indices": target.tolist(),
                    "prediction_indices": prediction.tolist(),
                })

    return {
        "loss": total_loss / total_plates,
        "character_accuracy": correct_characters / total_characters,
        "exact_plate_accuracy": exact_plates / total_plates,
        "position_accuracy": (position_correct.float() / total_plates).tolist(),
        "predictions": prediction_rows,
    }

def compact_metrics(prefix, metrics):
    return (
        f"{prefix} loss={metrics['loss']:.4f}, "
        f"character={metrics['character_accuracy']:.2%}, "
        f"exact={metrics['exact_plate_accuracy']:.2%}"
    )

In [ ]:
EPOCHS = 25
EARLY_STOPPING_PATIENCE = 7
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
BEST_RECOGNIZER_PATH = CHECKPOINT3_DIR / "seven_head_spatial_residual_best.pt"
TRAINING_HISTORY_PATH = RESULTS3_DIR / "training_history_spatial.csv"
REUSE_RECOGNIZER = (
    BEST_RECOGNIZER_PATH.is_file()
    and TRAINING_HISTORY_PATH.is_file()
)

optimizer = torch.optim.AdamW(
    recognizer.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

history = []
best_score = (-1.0, -1.0)
best_exact_accuracy = -1.0
best_character_accuracy = -1.0
epochs_without_improvement = 0

if REUSE_RECOGNIZER:
    saved_recognizer = torch.load(
        BEST_RECOGNIZER_PATH,
        map_location=DEVICE,
        weights_only=False,
    )
    recognizer.load_state_dict(saved_recognizer["model_state_dict"])
    best_exact_accuracy = float(
        saved_recognizer["validation_exact_plate_accuracy"]
    )
    print("Existing recognition checkpoint found:", BEST_RECOGNIZER_PATH)

for epoch in range(1, (0 if REUSE_RECOGNIZER else EPOCHS) + 1):
    start_time = time.perf_counter()
    train_metrics = run_epoch(recognizer, train_loader, optimizer=optimizer)
    validation_metrics = run_epoch(recognizer, validation_loader)
    scheduler.step(
        validation_metrics["exact_plate_accuracy"]
        + 0.01 * validation_metrics["character_accuracy"]
    )

    row = {
        "epoch": epoch,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "seconds": time.perf_counter() - start_time,
        "train_loss": train_metrics["loss"],
        "train_character_accuracy": train_metrics["character_accuracy"],
        "train_exact_plate_accuracy": train_metrics["exact_plate_accuracy"],
        "validation_loss": validation_metrics["loss"],
        "validation_character_accuracy": validation_metrics["character_accuracy"],
        "validation_exact_plate_accuracy": validation_metrics["exact_plate_accuracy"],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(TRAINING_HISTORY_PATH, index=False)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        + compact_metrics("train", train_metrics)
        + " | "
        + compact_metrics("validation", validation_metrics)
        + f" | {row['seconds']:.1f}s"
    )
    print(
        "Validation positions: "
        + ", ".join(
            f"P{position + 1}={accuracy:.2%}"
            for position, accuracy in enumerate(validation_metrics["position_accuracy"])
        )
    )

    current_score = (
        validation_metrics["exact_plate_accuracy"],
        validation_metrics["character_accuracy"],
    )
    if current_score > best_score:
        best_score = current_score
        best_exact_accuracy, best_character_accuracy = current_score
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": recognizer.state_dict(),
            "epoch": epoch,
            "validation_exact_plate_accuracy": best_exact_accuracy,
            "num_classes": NUM_CLASSES,
            "input_size": (PLATE_HEIGHT, PLATE_WIDTH),
        }, BEST_RECOGNIZER_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping: validation exact-plate accuracy did not improve.")
            break

print("Best checkpoint:", BEST_RECOGNIZER_PATH)
print("Best validation exact-plate accuracy:", f"{best_exact_accuracy:.2%}")

In [ ]:
history_df = pd.read_csv(RESULTS3_DIR / "training_history_spatial.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["validation_loss"], label="validation")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Training and validation loss")
axes[0].legend()
axes[1].plot(
    history_df["epoch"], history_df["validation_character_accuracy"], label="character"
)
axes[1].plot(
    history_df["epoch"], history_df["validation_exact_plate_accuracy"], label="exact plate"
)
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Validation accuracy", ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.show()

## Recognition results

These results use ground-truth plate boxes, which isolates the recogniser from detection errors. Exact-plate accuracy requires all seven characters to match.

In [ ]:
checkpoint = torch.load(BEST_RECOGNIZER_PATH, map_location=DEVICE, weights_only=False)
recognizer.load_state_dict(checkpoint["model_state_dict"])
recognizer.eval()

evaluation_frames = {"validation": validation_df, **test_frames}
evaluation_results = []
all_predictions = {}

for subset, frame in evaluation_frames.items():
    loader = DataLoader(
        CCPDRecognitionDataset(frame, evaluation_transform),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    metrics = run_epoch(recognizer, loader, collect_predictions=True)
    all_predictions[subset] = metrics["predictions"]
    row = {
        "subset": subset,
        "images": len(frame),
        "loss": metrics["loss"],
        "character_accuracy": metrics["character_accuracy"],
        "exact_plate_accuracy": metrics["exact_plate_accuracy"],
    }
    row.update({
        f"position_{position + 1}_accuracy": accuracy
        for position, accuracy in enumerate(metrics["position_accuracy"])
    })
    evaluation_results.append(row)
    print(compact_metrics(subset, metrics))

recognition_results_df = pd.DataFrame(evaluation_results)
recognition_results_df.to_csv(RESULTS3_DIR / "recognition_subset_results.csv", index=False)
display(recognition_results_df.style.format({
    "loss": "{:.4f}",
    "character_accuracy": "{:.2%}",
    "exact_plate_accuracy": "{:.2%}",
    **{f"position_{position}_accuracy": "{:.2%}" for position in range(1, 8)},
}))

## Recognition errors

In [ ]:
position_columns = [f"position_{position}_accuracy" for position in range(1, 8)]
test_result_rows = recognition_results_df[recognition_results_df["subset"].ne("validation")]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(test_result_rows["subset"], test_result_rows["exact_plate_accuracy"])
axes[0].set(title="Exact-plate accuracy by test subset", ylabel="Accuracy", ylim=(0, 1))
axes[0].tick_params(axis="x", rotation=35)
axes[1].bar(range(1, 8), recognition_results_df.iloc[0][position_columns])
axes[1].set(title="Validation accuracy by character position", xlabel="Position", ylabel="Accuracy", ylim=(0, 1))
plt.tight_layout()
plt.show()

base_predictions = pd.DataFrame(all_predictions["base"])
confusion = np.zeros((len(PROVINCES), len(PROVINCES)), dtype=int)
for ground_truth, prediction in zip(
    base_predictions["ground_truth_indices"], base_predictions["prediction_indices"]
):
    confusion[ground_truth[0], prediction[0]] += 1

plt.figure(figsize=(9, 8))
plt.imshow(confusion, cmap="Blues")
plt.colorbar(label="Images")
plt.xlabel("Predicted province index")
plt.ylabel("Ground-truth province index")
plt.title("Province confusion matrix on CCPD-Base")
plt.tight_layout()
plt.show()

example_groups = [
    ("Correct", base_predictions[base_predictions["errors"].eq(0)]),
    ("One error", base_predictions[base_predictions["errors"].eq(1)]),
    ("Multiple errors", base_predictions[base_predictions["errors"].ge(2)]),
]
fig, axes = plt.subplots(3, 2, figsize=(12, 7))
for row_index, (group_name, group) in enumerate(example_groups):
    examples = group.head(2).to_dict("records")
    for column_index, axis in enumerate(axes[row_index]):
        if column_index >= len(examples):
            axis.axis("off")
            continue
        example = examples[column_index]
        record = metadata_df.loc[
            metadata_df["relative_path"].eq(example["relative_path"])
        ].iloc[0]
        image = cv2.imread(str(Path(DATA_ROOT) / example["relative_path"]))
        crop = crop_bounding_box(
            image,
            (record["x_min"], record["y_min"], record["x_max"], record["y_max"]),
        )
        axis.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axis.set_title(
            f"{group_name} | GT {labels_to_display_text(example['ground_truth_indices'])} "
            f"| Pred {labels_to_display_text(example['prediction_indices'])}",
            fontsize=9,
        )
        axis.axis("off")
plt.tight_layout()
plt.show()

## Single-plate prediction

This function applies the trained recogniser to one image and one plate box. It returns the predicted indices and the confidence for each character.

In [ ]:
@torch.inference_mode()
def recognise_plate(image_bgr, bbox, model=recognizer):
    crop = crop_bounding_box(image_bgr, bbox)
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    input_tensor = evaluation_transform(crop_rgb).unsqueeze(0).to(DEVICE)
    outputs = model(input_tensor)
    indices = [int(output.argmax(dim=1).item()) for output in outputs]
    confidences = [float(output.softmax(dim=1).max().item()) for output in outputs]
    return {
        "text": labels_to_text(indices),
        "indices": indices,
        "character_confidences": confidences,
        "mean_confidence": float(np.mean(confidences)),
    }

example_row = test_frames["base"].iloc[0]
example_image = cv2.imread(str(Path(DATA_ROOT) / example_row["relative_path"]))
example_bbox = (
    example_row["x_min"], example_row["y_min"],
    example_row["x_max"], example_row["y_max"],
)
print("Ground truth:", example_row["plate_text"])
print("Recognition:", recognise_plate(example_image, example_bbox))

# Part 4: Detection and Recognition

The complete system uses the highest-confidence detector box as the input to the recogniser. Detected-box accuracy measures whether all seven characters are correct. Strict end-to-end accuracy also requires IoU above 0.7. Images with no detected plate are incorrect under both measures.

## Connecting the models

In [ ]:
from ultralytics import YOLO

PART2_HELPERS_DIR = PROJECT_DIR / "detection"
if str(PART2_HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(PART2_HELPERS_DIR))
from evaluate_detector import box_iou

END_TO_END_DIR = PROJECT_DIR / "pipeline"
END_TO_END_RESULTS_DIR = END_TO_END_DIR / "results"
END_TO_END_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DETECTOR_WEIGHTS = (
    PROJECT_DIR
    / "detection"
    / "runs"
    / "ccpd2019_yolov8n_gpu_30ep"
    / "weights"
    / "best.pt"
)
RECOGNIZER_WEIGHTS = (
    PROJECT_DIR
    / "recognition"
    / "checkpoints"
    / "seven_head_spatial_residual_best.pt"
)

DETECTOR_IMAGE_SIZE = 640
DETECTOR_BATCH_SIZE = 16
DETECTOR_CONFIDENCE = 0.25
PIPELINE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = PIPELINE_DEVICE
DETECTOR_DEVICE = 0 if PIPELINE_DEVICE.type == "cuda" else "cpu"
PIPELINE_SUBSETS = list(test_frames)

if not DETECTOR_WEIGHTS.is_file():
    raise FileNotFoundError(f"Detector checkpoint not found: {DETECTOR_WEIGHTS}")
if not RECOGNIZER_WEIGHTS.is_file():
    raise FileNotFoundError(f"Recognition checkpoint not found: {RECOGNIZER_WEIGHTS}")

detector_e2e = YOLO(str(DETECTOR_WEIGHTS))
recognizer_checkpoint = torch.load(
    RECOGNIZER_WEIGHTS,
    map_location=PIPELINE_DEVICE,
    weights_only=False,
)
recognizer.to(PIPELINE_DEVICE)
recognizer.load_state_dict(recognizer_checkpoint["model_state_dict"])
recognizer.eval()

def synchronise_device():
    if PIPELINE_DEVICE.type == "cuda":
        torch.cuda.synchronize()

def summarise_combined_predictions(predictions, subset, elapsed_seconds):
    images = len(predictions)
    detected = int(predictions["detected"].sum())
    localisation_correct = int(predictions["localisation_correct"].sum())
    exact_correct = int(predictions["exact_plate_correct"].sum())
    strict_correct = int((
        predictions["localisation_correct"]
        & predictions["exact_plate_correct"]
    ).sum())
    character_correct = int(predictions["character_correct"].sum())
    return {
        "subset": subset,
        "images": images,
        "detected": detected,
        "detection_rate": detected / images,
        "localisation_accuracy_iou_gt_0_7": localisation_correct / images,
        "recognition_accuracy_on_detected": exact_correct / detected if detected else 0.0,
        "end_to_end_character_accuracy": character_correct / (images * 7),
        "end_to_end_exact_plate_accuracy": exact_correct / images,
        "strict_end_to_end_accuracy": strict_correct / images,
        "elapsed_seconds": elapsed_seconds,
        "milliseconds_per_image": 1000 * elapsed_seconds / images,
        "fps": images / elapsed_seconds,
    }

def evaluate_combined_subset(subset, frame):
    records = []
    relative_paths = frame["relative_path"].tolist()
    resolved_paths = [str(Path(DATA_ROOT) / path) for path in relative_paths]

    synchronise_device()
    start_time = time.perf_counter()

    for start in range(0, len(frame), DETECTOR_BATCH_SIZE):
        stop = min(start + DETECTOR_BATCH_SIZE, len(frame))
        batch_frame = frame.iloc[start:stop]
        batch_results = detector_e2e.predict(
            source=resolved_paths[start:stop],
            imgsz=DETECTOR_IMAGE_SIZE,
            conf=DETECTOR_CONFIDENCE,
            batch=DETECTOR_BATCH_SIZE,
            device=DETECTOR_DEVICE,
            verbose=False,
            save=False,
        )
        if len(batch_results) != len(batch_frame):
            raise RuntimeError(
                f"Detector returned {len(batch_results)} results for "
                f"{len(batch_frame)} images."
            )

        for (_, row), result in zip(batch_frame.iterrows(), batch_results):
            ground_truth_box = np.asarray(
                [row["x_min"], row["y_min"], row["x_max"], row["y_max"]],
                dtype=float,
            )
            ground_truth_indices = parse_plate_indices(row["plate_indices"])
            predicted_box = None
            predicted_indices = []
            detector_confidence = 0.0
            overlap = 0.0
            mean_character_confidence = np.nan
            character_correct = 0
            exact_plate_correct = False

            if result.boxes is not None and len(result.boxes) > 0:
                confidences = result.boxes.conf.detach().cpu().numpy()
                best_index = int(np.argmax(confidences))
                detector_confidence = float(confidences[best_index])
                predicted_box = (
                    result.boxes.xyxy[best_index].detach().cpu().numpy().astype(float)
                )
                overlap = box_iou(ground_truth_box, predicted_box)
                image_bgr = result.orig_img
                if image_bgr is None:
                    image_bgr = cv2.imread(str(Path(DATA_ROOT) / row["relative_path"]))
                recognition = recognise_plate(
                    image_bgr,
                    predicted_box,
                    model=recognizer,
                )
                predicted_indices = recognition["indices"]
                mean_character_confidence = recognition["mean_confidence"]
                character_correct = sum(
                    predicted == target
                    for predicted, target in zip(predicted_indices, ground_truth_indices)
                )
                exact_plate_correct = predicted_indices == ground_truth_indices

            records.append({
                "subset": subset,
                "relative_path": row["relative_path"],
                "ground_truth_indices": "_".join(map(str, ground_truth_indices)),
                "prediction_indices": "_".join(map(str, predicted_indices)),
                "ground_truth_display": labels_to_display_text(ground_truth_indices),
                "prediction_display": (
                    labels_to_display_text(predicted_indices) if predicted_indices else ""
                ),
                "gt_x1": ground_truth_box[0],
                "gt_y1": ground_truth_box[1],
                "gt_x2": ground_truth_box[2],
                "gt_y2": ground_truth_box[3],
                "pred_x1": np.nan if predicted_box is None else predicted_box[0],
                "pred_y1": np.nan if predicted_box is None else predicted_box[1],
                "pred_x2": np.nan if predicted_box is None else predicted_box[2],
                "pred_y2": np.nan if predicted_box is None else predicted_box[3],
                "detected": predicted_box is not None,
                "detector_confidence": detector_confidence,
                "iou": overlap,
                "localisation_correct": overlap > 0.7,
                "character_correct": character_correct,
                "exact_plate_correct": exact_plate_correct,
                "mean_character_confidence": mean_character_confidence,
                "detector_inference_ms": float(result.speed.get("inference", np.nan)),
            })

        if stop % 200 == 0 or stop == len(frame):
            print(f"{subset}: {stop}/{len(frame)}")

    synchronise_device()
    elapsed_seconds = time.perf_counter() - start_time
    predictions = pd.DataFrame(records)
    if len(predictions) != len(frame):
        raise RuntimeError(f"Incomplete end-to-end results for {subset}.")
    summary = summarise_combined_predictions(
        predictions,
        subset,
        elapsed_seconds,
    )
    return predictions, summary

print("Detector checkpoint:", DETECTOR_WEIGHTS)
print("Recognition checkpoint:", RECOGNIZER_WEIGHTS)
print(
    "Pipeline device:",
    torch.cuda.get_device_name(0) if PIPELINE_DEVICE.type == "cuda" else "CPU",
)

## Combined results

In [ ]:
END_TO_END_PREDICTIONS_PATH = (
    END_TO_END_RESULTS_DIR / "end_to_end_predictions.csv"
)
END_TO_END_SUMMARY_PATH = END_TO_END_RESULTS_DIR / "end_to_end_summary.csv"

if END_TO_END_PREDICTIONS_PATH.is_file() and END_TO_END_SUMMARY_PATH.is_file():
    print("Existing end-to-end predictions found:", END_TO_END_PREDICTIONS_PATH)
    combined_predictions_df = pd.read_csv(END_TO_END_PREDICTIONS_PATH)
    for column in ("detected", "localisation_correct", "exact_plate_correct"):
        if combined_predictions_df[column].dtype != bool:
            combined_predictions_df[column] = (
                combined_predictions_df[column]
                .astype(str)
                .str.lower()
                .eq("true")
            )
    previous_summary_df = pd.read_csv(END_TO_END_SUMMARY_PATH)
    elapsed_by_subset = dict(zip(
        previous_summary_df["subset"],
        previous_summary_df["elapsed_seconds"],
    ))
    combined_summary_rows = [
        summarise_combined_predictions(
            combined_predictions_df[combined_predictions_df["subset"].eq(subset)],
            subset,
            float(elapsed_by_subset[subset]),
        )
        for subset in PIPELINE_SUBSETS
    ]
else:
    combined_prediction_frames = []
    combined_summary_rows = []
    for subset in PIPELINE_SUBSETS:
        predictions, summary = evaluate_combined_subset(subset, test_frames[subset])
        combined_prediction_frames.append(predictions)
        combined_summary_rows.append(summary)
    combined_predictions_df = pd.concat(
        combined_prediction_frames,
        ignore_index=True,
    )

subset_summary_df = pd.DataFrame(combined_summary_rows)
overall_summary = summarise_combined_predictions(
    combined_predictions_df,
    "overall",
    subset_summary_df["elapsed_seconds"].sum(),
)
combined_summary_df = pd.concat(
    [subset_summary_df, pd.DataFrame([overall_summary])],
    ignore_index=True,
)

combined_predictions_df.to_csv(END_TO_END_PREDICTIONS_PATH, index=False)
combined_summary_df.to_csv(END_TO_END_SUMMARY_PATH, index=False)

summary_columns = [
    "subset",
    "images",
    "detected",
    "detection_rate",
    "localisation_accuracy_iou_gt_0_7",
    "recognition_accuracy_on_detected",
    "end_to_end_exact_plate_accuracy",
    "strict_end_to_end_accuracy",
    "fps",
]
summary_display = combined_summary_df[summary_columns].copy()
for column in (
    "detection_rate",
    "localisation_accuracy_iou_gt_0_7",
    "recognition_accuracy_on_detected",
    "end_to_end_exact_plate_accuracy",
    "strict_end_to_end_accuracy",
):
    summary_display[column] = summary_display[column].map(lambda value: f"{value:.2%}")
summary_display["fps"] = summary_display["fps"].map(lambda value: f"{value:.2f}")
display(summary_display)

recognition_results_for_comparison = pd.read_csv(
    PROJECT_DIR
    / "recognition"
    / "results"
    / "recognition_subset_results.csv"
)
recognition_only = recognition_results_for_comparison[
    recognition_results_for_comparison["subset"].ne("validation")
][["subset", "images", "exact_plate_accuracy"]].rename(
    columns={"exact_plate_accuracy": "ground_truth_box_accuracy"}
)
comparison_df = recognition_only.merge(
    subset_summary_df[[
        "subset",
        "end_to_end_exact_plate_accuracy",
        "strict_end_to_end_accuracy",
    ]],
    on="subset",
    how="inner",
)
comparison_df["difference"] = (
    comparison_df["end_to_end_exact_plate_accuracy"]
    - comparison_df["ground_truth_box_accuracy"]
)
recognition_overall = np.average(
    comparison_df["ground_truth_box_accuracy"],
    weights=comparison_df["images"],
)
comparison_df = pd.concat([
    comparison_df,
    pd.DataFrame([{
        "subset": "overall",
        "images": int(comparison_df["images"].sum()),
        "ground_truth_box_accuracy": recognition_overall,
        "end_to_end_exact_plate_accuracy": overall_summary[
            "end_to_end_exact_plate_accuracy"
        ],
        "strict_end_to_end_accuracy": overall_summary[
            "strict_end_to_end_accuracy"
        ],
        "difference": (
            overall_summary["end_to_end_exact_plate_accuracy"] - recognition_overall
        ),
    }]),
], ignore_index=True)
comparison_df.to_csv(
    END_TO_END_RESULTS_DIR / "ground_truth_vs_detected_boxes.csv",
    index=False,
)

comparison_display = comparison_df.copy()
for column in (
    "ground_truth_box_accuracy",
    "end_to_end_exact_plate_accuracy",
    "strict_end_to_end_accuracy",
    "difference",
):
    comparison_display[column] = comparison_display[column].map(
        lambda value: f"{value:.2%}"
    )
display(comparison_display)

test_only = combined_summary_df[combined_summary_df["subset"].ne("overall")]
best_subset = test_only.loc[test_only["strict_end_to_end_accuracy"].idxmax()]
weakest_subset = test_only.loc[test_only["strict_end_to_end_accuracy"].idxmin()]
missed_images = int(overall_summary["images"] - overall_summary["detected"])
print(
    "Detected-box exact-plate accuracy:",
    f"{overall_summary['end_to_end_exact_plate_accuracy']:.2%}",
)
print(
    "Strict end-to-end accuracy:",
    f"{overall_summary['strict_end_to_end_accuracy']:.2%}",
)
print("Images with no detected plate:", missed_images)
print(
    "Highest strict subset accuracy:",
    best_subset["subset"],
    f"{best_subset['strict_end_to_end_accuracy']:.2%}",
)
print(
    "Lowest strict subset accuracy:",
    weakest_subset["subset"],
    f"{weakest_subset['strict_end_to_end_accuracy']:.2%}",
)
print("Complete-system speed:", f"{overall_summary['fps']:.2f} FPS")

## Failure cases

In [ ]:
failure_groups = [
    (
        "Correct",
        combined_predictions_df["localisation_correct"]
        & combined_predictions_df["exact_plate_correct"],
    ),
    (
        "Text correct, IoU <= 0.7",
        combined_predictions_df["detected"]
        & ~combined_predictions_df["localisation_correct"]
        & combined_predictions_df["exact_plate_correct"],
    ),
    ("Missed detection", ~combined_predictions_df["detected"]),
    (
        "Poor localisation",
        combined_predictions_df["detected"]
        & ~combined_predictions_df["localisation_correct"]
        & ~combined_predictions_df["exact_plate_correct"],
    ),
    (
        "Recognition error",
        combined_predictions_df["localisation_correct"]
        & ~combined_predictions_df["exact_plate_correct"],
    ),
]

failure_counts = pd.DataFrame([
    {"outcome": name, "images": int(mask.sum())}
    for name, mask in failure_groups
])
failure_counts.to_csv(
    END_TO_END_RESULTS_DIR / "outcome_counts.csv",
    index=False,
)
display(failure_counts)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis in axes.flat:
    axis.axis("off")
for axis, (group_name, mask) in zip(axes.flat, failure_groups):
    group = combined_predictions_df[mask]
    if group.empty:
        axis.text(0.5, 0.5, "No examples", ha="center", va="center")
        axis.set_title(group_name)
        axis.axis("off")
        continue

    example = group.sample(1, random_state=SEED).iloc[0]
    image = cv2.imread(str(Path(DATA_ROOT) / example["relative_path"]))
    if image is None:
        raise FileNotFoundError(example["relative_path"])

    ground_truth_box = tuple(
        int(example[column]) for column in ("gt_x1", "gt_y1", "gt_x2", "gt_y2")
    )
    cv2.rectangle(
        image,
        ground_truth_box[:2],
        ground_truth_box[2:],
        (0, 0, 255),
        3,
    )
    if example["detected"]:
        predicted_box = tuple(
            int(example[column])
            for column in ("pred_x1", "pred_y1", "pred_x2", "pred_y2")
        )
        cv2.rectangle(
            image,
            predicted_box[:2],
            predicted_box[2:],
            (0, 255, 0),
            3,
        )

    prediction_text = example["prediction_display"]
    if pd.isna(prediction_text) or not str(prediction_text).strip():
        prediction_text = "none"
    title = (
        f"{group_name}: {example['subset']}\n"
        f"GT {example['ground_truth_display']} | Pred {prediction_text}"
    )
    if example["detected"]:
        title += f" | IoU {example['iou']:.2f}"
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(title, fontsize=9)
    axis.axis("off")

plt.suptitle("End-to-end examples: ground truth red, prediction green")
plt.tight_layout()
failure_figure_path = END_TO_END_RESULTS_DIR / "failure_cases.png"
plt.savefig(failure_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("End-to-end results saved to:", END_TO_END_RESULTS_DIR)

## Discussion

The detector is the strongest part of the system. It finds a plate in 99.64% of the test images and reaches 90.46% localisation accuracy at IoU above 0.7. The complete pipeline runs at about 47 FPS on the test computer. Base and Weather also have very high plate-reading accuracy, which shows that the system works well when the plate is clear.

Recognition is the main source of error in the difficult subsets. Exact plate accuracy from detected boxes is 70.41%, while the strict end-to-end accuracy is 66.35%. Blur has the lowest result because several character shapes become difficult to separate. Dark or bright images, large rotation and perspective changes also reduce recognition accuracy. Only 36 images have no detection, so most complete-system errors happen after a plate region has already been found. There are also 406 images with correct text but IoU at or below 0.7, which is why both accuracy measures are reported.

The current recogniser assumes a fixed seven-character plate and the experiments use one plate per image. It is trained mainly with ground-truth boxes, so its input is not always the same as a detector crop. Training with more predicted boxes, stronger blur and lighting changes, and perspective correction could improve the difficult subsets. The experiment also uses the selected 45,000-image protocol rather than the complete CCPD benchmark, so the results should only be compared under the same data split and evaluation rules.